# 6.2 常見片段怎麼合併？


常見文字片段反覆出現，每次逐字母排成很多格有些浪費。可以先從細單位開始，把最常見的相鄰兩項黏成一個新項，再重複這個過程。這叫BPE，英文全名byte pair encoding；名稱源於byte對，但玩具示範也能從字元開始。這一步用片段次數訂出合併規則，供之後編碼文字使用。

用兩份文件 `abab` 與 `abac`。各自數相鄰符號：a-b共三次，b-a兩次，a-c一次，最常見的是a-b。因此把每份文件裡的非重疊a-b換成新符號ab，得到 `['ab','ab']` 與 `['ab','a','c']`。不能把第一份最後b與第二份開頭a當一對，因為那是兩個文件的邊界。

下圖上方兩列是兩份文件，四個框表示原本四項；a-b在第一列出現兩次、第二列一次，合計三次。中間列出三種相鄰對的次數，向下箭頭表示選出最多的ab，並把各列相鄰a、b合成一項。底下綠框的ab雖然寫兩個字母，現在只佔一個token，因此兩份文件分別剩兩項和三項。橘框標出要合併的a、b，第一列兩組、第二列一組；各組各自合併，不把整列黏成一項。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/foundations_bpe.svg")))

In [ ]:
from collections import Counter

corpus = [list("abab"), list("abac")]
pairs = Counter()
for row in corpus:
    for left, right in zip(row, row[1:]):
        pairs[(left, right)] += 1
pair = pairs.most_common(1)[0][0]


def merge(row, pair):
    out = []
    i = 0
    while i < len(row):
        if i + 1 < len(row) and tuple(row[i : i + 2]) == pair:
            out.append("".join(pair))
            i += 2
        else:
            out.append(row[i])
            i += 1
    return out


print(pairs)
print("選擇", pair, "結果", [merge(row, pair) for row in corpus])

輸入兩張字元清單。`row[1:]` 比原列晚一格，zip配出相鄰對，Counter數它們；`most_common(1)` 找最高頻那一項，再取其中的符號對。這裡a-b唯一最多，平手時則需規定穩定的選擇順序。計數與切片背景見 [1.4](https://birdhackor.github.io/tiny-perceptron-vlm/1.4.html)、[1.3](https://birdhackor.github.io/tiny-perceptron-vlm/1.3.html)。

`merge` 從左到右掃描。`while` 在條件成立時重複縮排代碼，i是現在位置；若還有下一項且兩項正是pair，就接成ab並跳兩格，否則複製當前項並只前進一格。`and` 要兩條件同時成立，先檢查不越界；`tuple` 把切片改成固定次序的小組，便於與pair比較。輸出四字的第一份縮成兩token，第二份縮成三token。

tokenizer是把原文切成片段、轉成ID並能還原文字的工具；訓練BPE tokenizer是在擬定這套合併規則，編碼則是拿已保存的規則處理文字，不是重新訓練語言模型。完整BPE會把新符號加入字表、重數新相鄰對，直到預算或停止條件滿足，並保存合併順序，編碼新文字時按已學規則操作。先分出一批用來訂規則的文字，叫訓練資料；另一批沒有參與訂規則的原文留作檢查，叫測試資料。某片段在訓練資料常見，通常更有機會合併。測試要看這套規則處理未見文字的表現；如果也用測試原文訂規則，就像事先照著考卷準備答案，成績會混入對這份考卷的特別安排。因此應先保留測試文字，只用訓練文字擬定規則。

合成的ab只表示這段文字現在一個token，不保證它是一個有語言意義的詞。若資料改變，最高頻對與後續拆法也可能變化；下一節用byte起點建立能覆蓋未見文字的版本。

練習再加一份 `list("acacacac")`，先預測a-c新增四次，總共五次，會勝過a-b的三次，再執行核對。觀察兩份原文怎麼按新pair合併，而不是隻看新增文件：tokenizer規則一變，同一舊文的編碼也會跟着變。
